# 05B Global Model Transfer Only

This notebook runs a strict saved-artifact transfer evaluation on M5 using synthetic-trained global boosting artifacts.

Important:
- this is an honest saved-model transfer test
- it does **not** retrain on M5
- it produces transfer metrics on monthly aggregated M5 data
- it is not the same as a Kaggle daily submission, because the saved synthetic artifacts are monthly models. This notebook is transfer evaluation only and does not generate uploadable XGBOOST/CATBOOST Kaggle CSVs.
- prerequisite: saved artifacts must already exist under `modeling/outputs/artifacts` (for example from notebook 02 global model training and artifact save)

In [4]:
from pathlib import Path
import pandas as pd

M5_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/external-data/m5-forecasting-accuracy')
REPORTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')
SCRIPT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/scripts/m5_saved_artifact_transfer.py')
TAG = 'strict_m5_transfer'

def resolve_report_path(kind: str, preferred_tag: str = TAG) -> Path:
    if not REPORTS_DIR.exists():
        raise FileNotFoundError(
            f'Reports directory not found: {REPORTS_DIR}. Run the transfer cell above first.'
        )

    preferred = REPORTS_DIR / f'{preferred_tag}_{kind}.csv'
    if preferred.exists():
        return preferred

    candidates = sorted(REPORTS_DIR.glob(f'*_{kind}.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
    if candidates:
        print(f"Preferred report not found: {preferred.name}. Using latest: {candidates[0].name}")
        return candidates[0]

    available = sorted(p.name for p in REPORTS_DIR.glob('*.csv'))
    raise FileNotFoundError(
        f"No '*_{kind}.csv' files found in {REPORTS_DIR}. Available CSVs: {available}"
    )

## Run Transfer Evaluation

In [8]:
ARTIFACTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/artifacts')
required_checks = [
    ARTIFACTS_DIR / 'A' / 'xgboost_h1' / 'production' / 'metadata.json',
    ARTIFACTS_DIR / 'A' / 'catboost_h1' / 'production' / 'metadata.json',
]
missing = [str(p) for p in required_checks if not p.exists()]

if missing:
    raise FileNotFoundError(
        'Missing saved model artifacts required for transfer evaluation. '
        'Run notebook 02_global_model_training_and_artifact_save.ipynb first. '
        f'Missing examples: {missing}'
    )

!python "{SCRIPT}" --m5-dir "{M5_DIR}" --granularity dept_store --datasets A B C --models XGBOOST CATBOOST --tag "{TAG}"

FileNotFoundError: Missing saved model artifacts required for transfer evaluation. Run notebook 02_global_model_training_and_artifact_save.ipynb first. Missing examples: ['/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/artifacts/A/xgboost_h1/production/metadata.json', '/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/artifacts/A/catboost_h1/production/metadata.json']

## Summary

In [7]:
summary_path = resolve_report_path('summary')
pd.read_csv(summary_path)

FileNotFoundError: No '*_summary.csv' files found in /Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports. Available CSVs: []

## Detailed Horizon Metrics

In [ ]:
metrics_path = resolve_report_path('metrics')
pd.read_csv(metrics_path).head(50)

## Why This Is Not a Kaggle Submission

The saved artifacts are monthly synthetic-trained models. Kaggle M5 submission requires 28-day daily item-store forecasts. That means:
- this notebook is valid for transfer evaluation
- it is not valid for strict Kaggle submission generation from the same saved monthly artifacts